# DengueWatch LK
### Explainable Machine Learning for Next-Week Dengue Case Forecasting in Sri Lanka

**Student No: 269161X**

This project predicts next week's dengue case count for Sri Lankan reporting areas and compares several machine learning models.

In [ ]:
!pip install -q xgboost shap lime joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('srilanka_weekly_data.csv')
df['date'] = pd.to_datetime(df['start.date'], format='%m/%d/%Y')
df = df.sort_values(['district','date']).reset_index(drop=True)

print(df.shape)
print(df['date'].min(), 'to', df['date'].max())
print('Reporting areas:', df['district'].nunique())

df.head()

## 1. Dataset

In [ ]:
print('Duplicate district-date rows:', df.duplicated(['district','date']).sum())
print(df['cases'].describe())

year_counts = df.groupby('year').size()
display(year_counts.tail(10))

df.groupby('district')['cases'].sum().sort_values(ascending=False).head(10)


In [ ]:
annual = df.groupby(df['date'].dt.year)['cases'].sum()
plt.figure(figsize=(10,4))
plt.plot(annual.index, annual.values, marker='o')
plt.title('Total Reported Dengue Cases by Year')
plt.xlabel('Year')
plt.ylabel('Cases')
plt.grid(alpha=0.2)
plt.show()


## 2. Feature Engineering

The target is the dengue case count for the next reporting week.

In [ ]:
grp = df.groupby('district')

for lag in [1,2,3,4,8,12,52]:
    df[f'cases_lag_{lag}'] = grp['cases'].shift(lag)

for w in [4,8,12]:
    df[f'rolling_mean_{w}'] = grp['cases'].transform(
        lambda s: s.shift(1).rolling(w).mean()
    )

df['trend_current_4'] = df['cases'] - df['cases_lag_4']
df['week_sin'] = np.sin(2*np.pi*df['week']/52.0)
df['week_cos'] = np.cos(2*np.pi*df['week']/52.0)

df['target_next_week'] = grp['cases'].shift(-1)
df['next_date'] = grp['date'].shift(-1)
df['gap_to_target'] = (df['next_date'] - df['date']).dt.days

ml = df[(df['gap_to_target'] >= 6) & (df['gap_to_target'] <= 9)].dropna().copy()

features = [
    'district','week','week_sin','week_cos','cases',
    'cases_lag_1','cases_lag_2','cases_lag_3','cases_lag_4',
    'cases_lag_8','cases_lag_12','cases_lag_52',
    'rolling_mean_4','rolling_mean_8','rolling_mean_12',
    'trend_current_4'
]

print('ML-ready rows:', len(ml))
ml[features + ['target_next_week']].head()


## 3. Train, Validation and Test Split

- Train: up to 2022
- Validation: 2023–2024
- Test: 2025

In [ ]:
train = ml[ml['date'].dt.year <= 2022].copy()
val = ml[ml['date'].dt.year.isin([2023, 2024])].copy()
test = ml[ml['date'].dt.year == 2025].copy()

print(len(train), len(val), len(test))


## 4. Model Comparison

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['district']),
    ('num', 'passthrough', [c for c in features if c != 'district'])
])

models = {
    'Decision Tree': DecisionTreeRegressor(
        max_depth=10, min_samples_leaf=10, random_state=42
    ),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=18, min_samples_leaf=2,
        max_features=0.8, n_jobs=-1, random_state=42
    ),
    'XGBoost': XGBRegressor(
        n_estimators=600, max_depth=4, learning_rate=0.03,
        subsample=0.9, colsample_bytree=0.9, min_child_weight=3,
        reg_lambda=2, objective='reg:squarederror',
        random_state=42, n_jobs=-1
    )
}


In [ ]:
rows = []
fitted = {}

for split_name, d in [('Validation', val), ('Test', test)]:
    y = d['target_next_week']
    pred = d['cases']
    rows.append([
        'Naive last-week', split_name,
        mean_absolute_error(y,pred),
        mean_squared_error(y,pred)**0.5,
        r2_score(y,pred)
    ])

for name, model in models.items():
    pipe = Pipeline([('preprocessor', pre), ('model', model)])
    pipe.fit(train[features], train['target_next_week'])
    fitted[name] = pipe

    for split_name, d in [('Validation', val), ('Test', test)]:
        pred = np.maximum(0, pipe.predict(d[features]))
        y = d['target_next_week']
        rows.append([
            name, split_name,
            mean_absolute_error(y,pred),
            mean_squared_error(y,pred)**0.5,
            r2_score(y,pred)
        ])

results = pd.DataFrame(rows, columns=['Model','Split','MAE','RMSE','R2'])
display(results.round(4))


### Model Result

The models are compared using MAE, RMSE and R².  
The best model is selected using the validation results.

The best model is selected using the validation results.

In [ ]:
ml_val = results[
    (results['Split']=='Validation') &
    (results['Model']!='Naive last-week')
].sort_values('RMSE')

best_name = ml_val.iloc[0]['Model']
best_model = fitted[best_name]

print('Selected ML model:', best_name)


## 5. Explainable AI with SHAP

SHAP is used to see which features influence the model predictions.

In [ ]:
import shap

pre_fitted = best_model.named_steps['preprocessor']
model_fitted = best_model.named_steps['model']

X_sample = test[features].sample(min(300, len(test)), random_state=42)
X_trans = pre_fitted.transform(X_sample)

feature_names = pre_fitted.get_feature_names_out()
clean_feature_names = [
    name.replace("num__", "").replace("cat__", "")
    for name in feature_names
]

explainer = shap.TreeExplainer(model_fitted)
shap_values = explainer(X_trans)

shap.summary_plot(
    shap_values.values,
    X_trans,
    feature_names=clean_feature_names
)

In [ ]:
shap_values_named = shap.Explanation(
    values=shap_values.values,
    base_values=shap_values.base_values,
    data=X_trans,
    feature_names=clean_feature_names
)

shap.plots.waterfall(shap_values_named[0], max_display=10)

### SHAP Interpretation

The SHAP plot shows how each feature affects the prediction.  
Current and recent dengue case counts have the strongest influence on the next-week forecast.

## 6. LIME

LIME is used to explain one individual prediction.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

X_train_trans = pre_fitted.transform(train[features])
X_test_trans = pre_fitted.transform(test[features])

lime_explainer = LimeTabularExplainer(
    X_train_trans,
    feature_names=clean_feature_names,
    mode='regression',
    random_state=42
)

lime_result = lime_explainer.explain_instance(
    X_test_trans[0],
    model_fitted.predict,
    num_features=10
)

lime_df = pd.DataFrame(
    lime_result.as_list(),
    columns=["Feature", "Contribution"]
)

lime_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(
    lime_df["Feature"],
    lime_df["Contribution"]
)

plt.xlabel("Contribution to prediction")
plt.ylabel("Feature")
plt.title("LIME Explanation for One Prediction")
plt.gca().invert_yaxis()
plt.show()

### LIME Interpretation

LIME explains one prediction locally by showing which feature conditions increase or decrease the forecast.